In [1]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
import os

In [2]:
# 使用小米 MiMo 模型
llm = ChatOpenAI(
    model="mimo-v2.5-pro",
    base_url="https://token-plan-cn.xiaomimimo.com/v1",
    api_key=os.environ.get("MIMO_API_KEY"),
    temperature=0
)

In [3]:
# --- Prompt 1: 提取信息 ---
prompt_extract = ChatPromptTemplate.from_template(
    "从以下文本中提取技术规格：\n\n{text_input}"
)

# --- Prompt 2: 转换为 JSON ---
prompt_transform = ChatPromptTemplate.from_template(
    "将以下规格转换为 JSON 对象，键为 'cpu'、'memory' 和 'storage'：\n\n{specifications}"
)

# --- 使用 LCEL 构建链 ---
# StrOutputParser() 将 LLM 的消息输出转换为简单字符串
extraction_chain = prompt_extract | llm | StrOutputParser()

# 完整链将提取链的输出传递给转换提示的 'specifications' 变量
full_chain = {"specifications": extraction_chain} | prompt_transform | llm | StrOutputParser()

# --- 运行链 ---
input_text = "这款新笔记本电脑配备 3.5 GHz 八核处理器、16GB 内存和 1TB NVMe 固态硬盘。"

# 使用输入文本字典执行链
final_result = full_chain.invoke({"text_input": input_text})

print("\n--- 最终 JSON 输出 ---")
print(final_result)


--- 最终 JSON 输出 ---
```json
{
  "cpu": "3.5 GHz 八核",
  "memory": "16GB",
  "storage": "1TB NVMe SSD"
}
```
